<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/02_Intermediate/L24_Instruction_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L24: Instruction Tuning - Teaching Models to Follow Instructions

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Intermediate  
**Lesson:** 24 of 30

---

## Learning Objectives

By the end of this lesson, you will:
- Format instruction-response pairs for supervised fine-tuning (SFT)
- Fine-tune a causal LM on instruction data using the Trainer
- Use chat templates (e.g. for Llama) when available

---

## Concept: Instruction Tuning

**Instruction tuning** fine-tunes a language model on (instruction, response) pairs so it learns to follow user directives. Data format: "Instruction: ...\nResponse: ..." or use a chat template. We use a small synthetic dataset and train GPT-2 (or a small decoder) to generate the response given the instruction.

---

In [ ]:
!pip install transformers torch datasets -q
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Instruction Dataset Format

---

In [ ]:
instructions = [
    "What is 2+2?", "Explain what ML is.", "Write a short greeting.", "Translate hello to Spanish.",
    "What is the capital of France?", "List two colors.", "Say true or false: Earth is round.",
] * 25
responses = [
    "4", "Machine learning is a type of AI that learns from data.", "Hello! How can I help?", "Hola.",
    "Paris.", "Red and blue.", "True.",
] * 25

def format_instruction(inst, resp):
    return f"Instruction: {inst}\nResponse: {resp}"

texts = [format_instruction(i, r) for i, r in zip(instructions, responses)]
dataset = Dataset.from_dict({"text": texts})
print("Sample:", texts[0][:80] + "...")

## Part 2: Tokenize for Causal LM

We tokenize the full sequence; the model is trained to predict the next token (so it learns to generate "Response: ..." after "Instruction: ...").

---

In [ ]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    out = tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out

dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
dataset.set_format("torch")
print("Dataset size:", len(dataset))

## Part 3: Train Causal LM on Instruction Data

---

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)
args = TrainingArguments(
    output_dir="./out_instruct_l24",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=dataset)
trainer.train()
print("Instruction tuning done. Prompt with 'Instruction: ...\\nResponse:' and generate.")

## Part 4: Test Instruction Following

---

In [ ]:
prompt = "Instruction: What is 2+2?\nResponse:"
inputs = tokenizer(prompt, return_tensors="pt")
out = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print("Generated:", tokenizer.decode(out[0], skip_special_tokens=True))

## Exercises

1. Add 20+ diverse instruction-response pairs and re-train; compare output quality.
2. Use a chat template (e.g. for a model that supports it) and format data accordingly.
3. Mask loss on only the response tokens (keep instruction tokens with label -100) for better efficiency.

---

## Key Takeaways

1. Instruction tuning = SFT on (instruction, response) pairs; format as single text or use chat template.
2. Causal LM learns to continue after "Instruction: ..." with the response.
3. For production, use larger base models and larger instruction datasets (e.g. Alpaca, ShareGPT).

---

## Next Lesson

**L25: RLHF Basics** – Reward models and reinforcement learning from human feedback.

---